## Problem 1

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Problem 2
Omega_ss = 2*np.pi/365/(24*3600)
print(Omega_ss)
mu = 398600
J2 = 1.08263e-3
R0 = 6378
elist = [0,0.2,0.4,0.6,0.8,0.9]

def inc(a,am):
    """ 
    Calculate inclination as a function of radius
    """
    i = np.arccos(-a/am)
    return i



for j in range(len(elist)):
    e = elist[j]
    am = (3/2*np.sqrt(mu)*J2*R0**2/(1-e**2)) 
    a = np.linspace(6400,am,1000)
    i = [inc(x,am) for x in a]
    i = [x*180/np.pi for x in i]
    plt.plot(a,i,label=f'e={elist[j]}')



plt.grid()
plt.xlabel('Semimajor Axis (km)')
plt.ylabel('Inclination (deg)')
plt.title('Inclination vs Semimajor Axis')
plt.legend()
plt.show()

Tmax = 2*np.pi/np.sqrt(mu)*am**(3/2)
Tmin = 2*np.pi/np.sqrt(mu)*R0**(3/2)

print(f'Tmax = {Tmax} s, Tmin = {Tmin}')
print(f'am = {am}')



In [ ]:
# Problem 1.d
import numpy as np
from ..orbit_package.orbit_package import Orbit
orb = Orbit()

T = 12 # hrs
T = 12*3600 # s
omega = -np.pi/2
mu = 398600 #km^3/s^2
J2 = 1.08263e-3
R = 6378
rp = 7000 #km

# Finding SMA from T
a = (T*np.sqrt(mu)/(2*np.pi))**(2/3)

# Finding eccentricity
e = 1-rp/a

# Finding orbit rate
cosi = 1/np.sqrt(5)
Omega = -3/2*np.sqrt(mu)*J2*R**2/(1-e**2)**2/a**(7/2)*cosi
print(f'Omega_dot = {Omega} rad/s = {Omega*180/np.pi*(3600*24)} deg/day')

# Part ii: Percent of time in northern hemisphere
f1 = np.pi/2
f2 = 3*np.pi/2

t = orb.tof_two_nu(f1,f2,a,e,mu)
print(f'Time in northern hemisphere: t = {t} s')
print(f'Time in southern hemisphere: t = {T - t} s')
print(f'Percentage of time in Norhern Hemisphere: {t/T*100}%')

# Part iii: argp drift
i = np.arctan2(2/np.sqrt(5),1/np.sqrt(5))
i = i - 0.5*np.pi/180
omega_dot = -3/2*(np.sqrt(mu)*J2*R**2)/((1-e**2)**2*a**(7/2))*(5/2*np.sin(i)**2 - 2)
t_drift = np.pi/omega_dot
print(f't_drift = {t_drift} s = {t_drift/3600/24/365} years')

In [ ]:
# Probem 1.e
import numpy as np
from ..orbit_package.orbit_package import Orbit
orb = Orbit()
import matplotlib.pyplot as plt

# Givens
e = 0
a = 6578
mu = 398600
n = np.sqrt(mu/a**3)
J2 = 1.08263e-3
R = 6378

i_list = [0,np.pi/2]

for i in i_list:
    print('')
    print(f'i = {i} rad: ')
    pv = orb.j2_perturbation(e,i,a)
    print(pv)

    # Finding radial difference solutions
    dn = np.abs(pv['n'] - pv['nbar'])  
    print(f'dn = {dn}')
    
    # time to separate by pi rad
    t = np.pi/dn
    print(f'The orbits will be separated 180deg after {t} seconds = {t/3600/24} days')

    # AT distance between them
    T = 2*np.pi/np.sqrt(mu)*a**(3/2)
    df = T*dn

    dr = df*a
    print(f'df = {df} rad, dr_at = {dr} km')






## Problem 2

### Part a) Implement J2 effect in the numerical propagator

In [ ]:
import numpy as np
from orbit_package.numerical import Integrator, ForcingFunction
from orbit_package.graphing_utils import Graph
from orbit_package.ephemeris import Ephemeris
from orbit_package.orbit_package import Orbit

# Instantiating all obejects
integrate = Integrator()
ff = ForcingFunction()
gr = Graph()
orb = Orbit()

# Initial conditions and parameters
x0 = np.array([6e3,6e3,6e3,-5,5,0])
other = {"mu":4e5,"J2":1.08263,"R":6378}
period = 17933.95497
tspan = [0,5*period]
integration_time = np.linspace(0,2*int(period),int(2*int(period)/60))

# Integrating
t,x = integrate.ode45(ff.twobody_j2_nodyn,x0,tspan,other)

eph = integrate.eph_from_propagation_results(t,x,"2BODY_J2")
eph = orb.fill_eph_osculating(eph,other['mu'])

gr.plot_eph_osc(eph,other['mu'])
gr.plot_eph_pos_vector(eph,earth=True,gradient=True)

# Determining that the following are constant
# Angular Momentum about the z axis:
hvecs = eph.all_h_osc()
z = np.array([0,0,1])
hz = [np.dot(hvec,z) for hvec in hvecs]
gr.plot_data(hz,x=t,xlab='Time (s)',ylab='Angular Momentum (km^2/s)',title='Angular Momentum vs Time',Name='Angular Momentum')

# Enery
vlist = eph.all_v_norm()
rlist = eph.all_r_norm()
zlist = eph.all_z()
E = []
for i in range(len(eph.data)):
    E.append(1/2*vlist[i]**2 - other['mu']/rlist[i] - other['mu']/(2*rlist[i]**3)*other['R']**2*other['J2']*(1-3*zlist[i]**2/rlist[i]**2))

gr.plot_data(E,x=t,xlab='Time (s)',ylab='Energy (km^2/s^2)',title='Orbit Energy vs Time',Name='Orbit Energy')

ModuleNotFoundError: No module named 'orbit_package'

### Implementing the updated code to propagate an orbit under J2 perturbation

In [ ]:
import numpy as np
from ..orbit_package.numerical import Integrator, ForcingFunction
from ..orbit_package.graphing_utils import Graph
from ..orbit_package.ephemeris import Ephemeris
from ..orbit_package.orbit_package import Orbit

# Instantiating all obejects
integrate = Integrator()
ff = ForcingFunction()
gr = Graph()
orb = Orbit()

# # Initial conditions and parameters
a = 7000.0
e = 0.1
i = np.pi/4
omega = 0.0      # argument of perigee
Omega = 0.0      # RAAN
E = 0.0          # eccentric anomaly (rad)
mu = 398600.0
a0 = 7000
sigma = 0
T = 2*np.pi*a**(3/2)/np.sqrt(mu)
n = np.sqrt(mu/a**3)
n0 = n
J2 = 1.08263e-3
R = 6378
elements = {"sma":a,"ecc":e,"E":E,"argp":omega,"raan":Omega,"inc":i}
r_vec, v_vec = orb.kep_to_cart(elements,mu)
x0 = np.concat([r_vec,v_vec])
tspan = [0,5*T]

# Integrating
other = {"mu":mu,"J2":1.08263e-3,"R":6378}
tout,x = integrate.ode45(ff.twobody_j2_nodyn,x0,tspan,other)

eph = integrate.eph_from_propagation_results(tout,x,"2BODY_J2")
eph = orb.fill_eph_osculating(eph,other['mu'])

# Solvng LPE for Averages

abar = [a]
ebar = [e]
ibar = [i]
omegabar = [omega]
Omegabar = [Omega]
Mbar = [0]
nbar = [n]
for t in tout:
    a = abar[-1]
    i = ibar[-1]
    e = ebar[-1]
    p = a*(1-e**2)
    
    # Performing first order averaging
    nbar.append(n)
    abar.append(a)
    ebar.append(e)
    ibar.append(i)
    omegabar.append(omegabar[0] + 3/2*J2*R**2/p**2*nbar[-1]*(2 - 5/2*np.sin(i)**2)*t)
    Omegabar.append(Omegabar[0] - 3/2*J2*R**2/p**2*nbar[-1]*np.cos(i)*t)
    Mbar.append(Mbar[0] + nbar[-1]*t)

averaged = {'nbar':nbar,'abar':abar,'ebar':ebar,'ibar':ibar,'omegabar':omegabar,'Omegabar':Omegabar,'Mbar':Mbar}

gr.plot_eph_osc_withJ2(eph,averaged,mu)

In [ ]:
i0 = np.pi/4
e0 = 0.1
# Comparing LPE and true averages
a_avg = np.mean(eph.all_sma_osc())
i_avg = np.mean(eph.all_inc_osc())
e_avg = np.mean(eph.all_ecc_osc())
n_avg = np.mean(eph.all_n_osc())

# Difference between eph and each average at each point
a = eph.all_sma_osc()
diff_a_avg = a - a_avg
diff_a_lpe = a - np.float64(a0)
avg_diff_a_avg = np.mean(diff_a_avg)
avg_diff_a_lpe = np.mean(diff_a_lpe)
print(f'Average diff a_avg: {avg_diff_a_avg}, Average diff a_lpe: {avg_diff_a_lpe}')

i = eph.all_inc_osc()
diff_i_avg = i - i_avg
diff_i_lpe = i - np.float64(i0)
avg_diff_i_avg = np.mean(diff_i_avg)
avg_diff_i_lpe = np.mean(diff_i_lpe)
print(f'Average diff i_avg: {avg_diff_i_avg}, Average diff i_lpe: {avg_diff_i_lpe}')

e = eph.all_ecc_osc()
diff_e_avg = e - e_avg
diff_e_lpe = e - np.float64(e0)
avg_diff_e_avg = np.mean(diff_e_avg)
avg_diff_e_lpe = np.mean(diff_e_lpe)
print(f'Average diff e_avg: {avg_diff_e_avg}, Average diff e_lpe: {avg_diff_e_lpe}')

n = eph.all_n_osc()
diff_n_avg = n - n_avg
diff_n_lpe = n - np.float64(n0)
avg_diff_n_avg = np.mean(diff_n_avg)
avg_diff_n_lpe = np.mean(diff_n_lpe)
print(f'Average diff n_avg: {avg_diff_n_avg}, Average diff n_lpe: {avg_diff_n_lpe}')


In [ ]:
# Solving LPE for initial conditions
import matplotlib.pyplot as plt
import numpy as np

a = 7000.0
e = 0.1
i = np.pi/4
omega = np.pi/3      # argument of perigee
Omega = np.pi/2     # RAAN
E = np.pi   
f = np.pi 
mu = 398600.0
sigma = 0
T = 2*np.pi*a**(3/2)/np.sqrt(mu)
n = np.sqrt(mu/a**3)
J2 = 1.08263e-3
R = 6378
elements = {"sma":a,"ecc":e,"E":E,"argp":omega,"raan":Omega,"inc":i}
r_vec, v_vec = orb.kep_to_cart(elements,mu)
print(f'r = {r_vec}')
print(f'v = {v_vec}')

rnorm = a*(1-e**2)/(1+e*np.cos(f))

x = rnorm*(np.cos(Omega)*np.cos(Omega+f) - np.sin(Omega)*np.sin(omega + f)*np.cos(i))
print(x)

## Problem 3: Numerically Integrating Solar Radiation Pressure

In [ ]:
import numpy as np
from ..orbit_package.numerical import Integrator, ForcingFunction
from ..orbit_package.graphing_utils import Graph
from ..orbit_package.ephemeris import Ephemeris
from ..orbit_package.orbit_package import Orbit

# Instantiating all obejects
integrate = Integrator()
ff = ForcingFunction()
gr = Graph()
orb = Orbit()

# # Initial conditions and parameters
a = 1
e = 0
i = 0
omega = 0.0      # argument of perigee
Omega = 0.0      # RAAN
E = 0.0          # eccentric anomaly (rad)
mu = 1
T = 2*np.pi*a**(3/2)/np.sqrt(mu)
n = np.sqrt(mu/a**3)

elements = {"sma":a,"ecc":e,"E":E,"argp":omega,"raan":Omega,"inc":i}
r_vec, v_vec = orb.kep_to_cart(elements,mu)

x0 = np.concat([r_vec,v_vec])
tspan = [0,5*T]

# Integrating
g = 0.01
other = {"mu":mu,'g':g}
tout,x = integrate.ode45(ff.twobody_srp_nodyn,x0,tspan,other)

eph = integrate.eph_from_propagation_results(tout,x,"2BODY_SRP")
eph = orb.fill_eph_osculating(eph,other['mu'])
gr.plot_eph_osc(eph,mu)
gr.plot_eph_pos_vector(eph,earth=True,r_earth=0.1,gradient=True)


# Ensuring constant characterstics are met
h_vecs = eph.all_h_osc('vec')
hx = [h[0] for h in h_vecs]
gr.plot_data(hx,x=tout,xlab='Time',ylab='Specific Angular Momentum (x) [distance^2/time]',title='Specific Angular Momentum About x Axis',Name='Eph Angular Momentum')

v = eph.all_v_norm()
r = eph.all_r_norm()
x = eph.all_x()

E = []
for j in range(len(x)):
    vj = v[j]
    rj = r[j]
    xj = x[j]
    E.append(0.5*vj**2 - mu/rj - g*xj)

gr.plot_data(E,x=tout,xlab='Time',ylab='Specific Energy [distance^2/time^2]',title='Orbit Energy vs Time',Name='Eph Energy')


In [ ]:
abar = [a]
ebar = [0]
ibar = [0]
sigmabar = [0]
omegatildebar = [3*np.pi/2]

for t in tout:
    # Extracting old values, calculating n
    a = abar[-1]
    n = np.sqrt(mu/a**3)
    e = ebar[-1]
    i = ibar[-1]
    sigma = sigmabar[-1]
    omegatilde = omegatildebar[-1]

    # Calclating new values
    eb = ebar[0] - 3*g/(2*n*a)*np.sqrt(1-e**2)*np.sin(omegatilde)*t
    ebar.append(eb)

    abar.append(a)
    ibar.append(i)

    sigmab = sigmabar[0] + np.sin(omegatilde)*(3/2*(1-e**2)*g/(n*a*e)+3*g*e/(n*a))*t
    sigmabar.append(sigmab)

    # omegatildeb = omegatildebar[0] - 3/2*g*np.sqrt(1-e**2)/(n*a*e)*np.cos(omegatilde)
    omegatildeb = omegatildebar[0]
    omegatildebar.append(omegatildeb)


omegatildebar = [x*180/np.pi for x in omegatildebar]
averaged = {'abar':abar,'ebar':ebar,'ibar':ibar,'sigmabar':sigmabar,'omegatildebar':omegatildebar}
gr.plot_data(omegatildebar)
gr.plot_eph_osc_withsrp(eph,averaged,mu)

In [ ]:
# Plotting the results for all g
# Integrating
glist = [0.001,0.01,0.1,1]
glist_labels = ['g=0.001','g=0.01','g=0.1','g=1']
eph_list = []
for g in glist:
    other = {"mu":mu,'g':g}
    tout,x = integrate.ode45(ff.twobody_srp_nodyn,x0,tspan,other)

    eph = integrate.eph_from_propagation_results(tout,x,"2BODY_J2")
    eph = orb.fill_eph_osculating(eph,other['mu'])
    eph_list.append(eph)
    #gr.plot_eph_osc(eph,mu)
    gr.plot_eph_pos_vector(eph,earth=True,r_earth=0.1,gradient=True)

#gr.plot_eph_pos_vector_listeph(eph_list,gradient = True,legend_values = glist_labels)

In [ ]:
from ASEN 5050.orbit_package.orbit_package import Orbit
from orbit_package.graphing_utils import Graph

orb = Orbit()
gr = Graph()
x0 = np.array([6e3,6e3,6e3,-5,5,0])
period = 17933.95497
tspan = [0,5*period]
other = {"mu":398600,'g':0.001,'J2':1.08263e-3,'R':6378}
tout,x = integrate.ode45(ff.twobody_j2srp_nodyn,x0,tspan,other)

eph = integrate.eph_from_propagation_results(tout,x,"2BODY_J2")
eph = orb.fill_eph_osculating(eph,other['mu'])
gr.plot_eph_osc(eph,mu)
gr.plot_eph_pos_vector(eph,earth=True,r_earth=0.1,gradient=True)




ModuleNotFoundError: No module named 'orbit_package'